# H1 charged-particle $p_T^*$ spectra from reweighted DIS events

This notebook processes the schema-v4 shower-on event sample
`results/dis_dijet_tmd_output.txt` and compares it with the H1 measurement
**Measurement of Charged Particle Spectra in Deep-Inelastic ep Scattering at HERA**
(H1 Collaboration, arXiv:1302.1321), Tables 8 and 9 in the supplied `H1Pt.pdf`.

The target observable is

$$
\frac{1}{N}\frac{dn}{dp_T^*}
$$

for $5<Q^2<10\;\mathrm{GeV}^2$, $0.05<y<0.6$, three Bjorken-$x$ bins,
and the two HCM regions $0<\eta^*<1.5$ and $1.5<\eta^*<5$.

Two distinct momentum fractions are kept separate throughout:

- **$x_B$ (the paper's $x$)** selects the H1 event bin.
- **$x_g$** is supplied to the elastic/inelastic hard cross-section functions.

No unit-area normalization or post-hoc data matching is applied. The comparison is an
absolute weighted charged-particle multiplicity per selected DIS event.


## 1. Configure paths, numerical precision, and the GPU

Cross-section inference and the large flattened-hadron operations use CUDA whenever it
is available. Text parsing remains on the CPU because it is an I/O task. The radial
quadrature order is fixed before importing `corss_section.py` so cache identities and
reruns are reproducible.


In [1]:
from pathlib import Path
import gc
import hashlib
import json
import math
import os
import re
import sys
import time
import warnings

# Resolve paths relative to the notebook working directory (project root).
PROJECT_DIR = Path(".").resolve()
DIS_DIR = PROJECT_DIR / "DIS"
os.environ["MPLCONFIGDIR"] = str(PROJECT_DIR / ".matplotlib-emulator")
os.environ["DIJET_EMULATOR_DEVICE"] = "cuda"
os.environ.setdefault("CROSS_SECTION_N_RADIAL", "64")
# Keep dipole NN batches small when the GPU is shared / fragmented.
os.environ.setdefault("CROSS_SECTION_DIPOLE_BATCH_SIZE", "128")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

EVENT_FILE = DIS_DIR / "results" / "dis_dijet_tmd_output.txt"
PAPER_FILE = PROJECT_DIR / "H1Pt.pdf"
OUTPUT_DIR = DIS_DIR / "results" / "H1"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

M_PROTON = 0.938
E_ELECTRON = 27.6
E_PROTON = 920.0
H1_Q2_RANGE = (5.0, 10.0)
H1_Y_RANGE = (0.05, 0.6)
XG_CROSS_SECTION_CAP = 0.01
LARGE_X_SUPPRESSION_POWER = 4
# Outer event batch for el/inel cross sections; reduce further if OOM persists.
WEIGHT_BATCH_SIZE = int(os.environ.get("H1_WEIGHT_BATCH_SIZE", "10000"))
X_BINS = (
    (1.0e-4, 2.4e-4),
    (2.4e-4, 5.0e-4),
    (5.0e-4, 2.0e-3),
)

if WEIGHT_BATCH_SIZE <= 0:
    raise ValueError("H1_WEIGHT_BATCH_SIZE must be positive.")

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda":
    warnings.warn(
        "CUDA is unavailable; the notebook will run on CPU and may be slow.",
        RuntimeWarning,
    )
else:
    torch.cuda.set_device(DEVICE)


def release_cuda_memory(move_dipole_to_cpu=False, move_emulator_to_cpu=False):
    """Release unused tensors and optionally move persistent models to the CPU."""
    if move_dipole_to_cpu:
        module = sys.modules.get("dipole_function")
        if module is not None and hasattr(module, "models_dict"):
            module.models_dict.to("cpu")
            integrator = getattr(module, "phys_integrator", None)
            if integrator is not None:
                integrator.to("cpu")
                integrator.device = torch.device("cpu")
            module.device = torch.device("cpu")
    if move_emulator_to_cpu:
        module = sys.modules.get("dijet_emulator.dijet_emulator")
        if module is not None and hasattr(module, "_model"):
            module._model.to("cpu")
            for name in ("_x_mean", "_x_std", "_y_mean", "_y_std"):
                tensor = getattr(module, name, None)
                if torch.is_tensor(tensor):
                    setattr(module, name, tensor.cpu())
            module.device = torch.device("cpu")
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.synchronize(DEVICE)
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except RuntimeError:
            pass


plt.rcParams.update(
    {
        "figure.dpi": 120,
        "savefig.dpi": 220,
        "font.size": 11,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "grid.linestyle": ":",
    }
)

print(f"Python executable: {sys.executable}")
print(f"PyTorch: {torch.__version__}")
print(f"Compute device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(DEVICE)}")
    free_b, total_b = torch.cuda.mem_get_info(DEVICE)
    print(
        f"CUDA free/total at startup: "
        f"{free_b / (1024**3):.2f}/{total_b / (1024**3):.2f} GiB"
    )
print(f"Event file: {EVENT_FILE}")
print(f"H1 paper: {PAPER_FILE}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Weight batch size: {WEIGHT_BATCH_SIZE}")
print(f"Dipole batch size: {os.environ['CROSS_SECTION_DIPOLE_BATCH_SIZE']}")


Python executable: /home/congyi/anaconda3/envs/emulator/bin/python
PyTorch: 2.13.0+cu130
Compute device: cuda:0
GPU: NVIDIA GeForce RTX 5080
CUDA free/total at startup: 2.03/15.47 GiB
Event file: /home/congyi/work/dipole_function_saved/DIS/results/dis_dijet_tmd_output.txt
H1 paper: /home/congyi/Appinstall/NutSync/Nutstore/博士后/pythia8/experiment/H1Pt.pdf
Output directory: /home/congyi/work/dipole_function_saved/DIS/results/H1
Weight batch size: 10000
Dipole batch size: 128


## 2. Encode the H1 measurements from Tables 8 and 9

The experimental central values have units $\mathrm{GeV}^{-1}$. The published
statistical and systematic errors are relative percentages. For plotting, the total
point-to-point uncertainty is

$$
\delta_{\rm H1}
= y_{\rm H1}\sqrt{
  \left(\frac{\delta_{\rm stat}}{100}\right)^2+
  \left(\frac{\delta_{\rm sys}}{100}\right)^2}.
$$

Systematic correlations are not supplied by the PDF, so any diagonal $\chi^2$ reported
later is a diagnostic rather than a full experimental likelihood.


In [2]:
def measurement(edges, values, stat_percent, sys_percent):
    """Build and validate one H1 binned measurement."""
    result = {
        "edges": np.asarray(edges, dtype=np.float64),
        "values": np.asarray(values, dtype=np.float64),
        "stat_percent": np.asarray(stat_percent, dtype=np.float64),
        "sys_percent": np.asarray(sys_percent, dtype=np.float64),
    }
    n_bins = len(result["edges"]) - 1
    for key in ("values", "stat_percent", "sys_percent"):
        if len(result[key]) != n_bins:
            raise ValueError(f"Inconsistent H1 bin count for {key}.")
    if np.any(np.diff(result["edges"]) <= 0.0):
        raise ValueError("H1 pT edges must be strictly increasing.")
    if np.any(result["values"] <= 0.0):
        raise ValueError("H1 densities must be positive.")
    return result


H1_MEASUREMENTS = {
    "central": (
        measurement(
            [0.2, 0.4, 0.6, 1.0, 2.0, 4.0, 10.0],
            [4.76, 2.92, 1.144, 0.1955, 1.489e-2, 5.69e-4],
            [0.03, 0.04, 0.10, 0.39, 1.89, 15.36],
            [3.2, 2.9, 3.7, 3.0, 3.2, 6.0],
        ),
        measurement(
            [0.2, 0.4, 0.6, 1.0, 2.0, 4.0, 10.0],
            [3.99, 2.53, 0.994, 0.1611, 1.286e-2, 5.1e-4],
            [0.05, 0.06, 0.14, 0.52, 2.58, 21.40],
            [2.5, 2.6, 2.9, 3.3, 2.7, 4.9],
        ),
        measurement(
            [0.2, 0.6, 1.0, 2.0, 4.0, 10.0],
            [2.097, 0.659, 0.1113, 8.82e-3, 3.3e-4],
            [0.11, 0.19, 0.74, 3.40, 34.47],
            [3.0, 3.4, 2.8, 3.1, 5.5],
        ),
    ),
    "current": (
        measurement(
            [0.0, 0.3, 0.6, 1.0, 1.5, 2.1, 3.0, 4.0, 5.0, 6.3, 7.9, 10.0],
            [5.60, 6.83, 2.290, 0.639, 0.1897, 5.12e-2,
             1.378e-2, 3.97e-3, 1.40e-3, 4.34e-4, 9.3e-5],
            [0.04, 0.04, 0.07, 0.14, 0.30, 0.69,
             1.40, 2.46, 4.68, 9.33, 20.80],
            [2.5, 2.8, 3.6, 3.3, 2.7, 4.0,
             4.8, 8.6, 9.7, 6.2, 13.2],
        ),
        measurement(
            [0.0, 0.3, 0.6, 1.0, 1.5, 2.1, 3.0, 4.0, 5.0, 6.3, 7.9, 10.0],
            [5.57, 6.64, 2.207, 0.587, 0.161, 4.12e-2,
             1.002e-2, 2.63e-3, 7.69e-4, 2.46e-4, 6.65e-5],
            [0.04, 0.04, 0.08, 0.17, 0.36, 0.85,
             1.83, 3.46, 7.08, 13.98, 32.90],
            [2.8, 2.6, 2.2, 1.7, 1.8, 3.2,
             5.7, 6.7, 5.7, 10.4, 18.6],
        ),
        measurement(
            [0.0, 0.3, 0.6, 1.0, 1.5, 2.1, 3.0, 4.0, 5.0, 6.3, 7.9],
            [4.96, 5.74, 1.863, 0.472, 0.1214, 2.85e-2,
             6.14e-3, 1.55e-3, 4.401e-4, 1.11e-4],
            [0.05, 0.04, 0.08, 0.19, 0.40, 1.02,
             2.40, 4.94, 10.49, 24.17],
            [2.2, 2.1, 3.4, 2.4, 2.5, 5.5,
             6.1, 5.4, 12.8, 13.6],
        ),
    ),
}

h1_rows = []
for region, measurements in H1_MEASUREMENTS.items():
    for x_index, item in enumerate(measurements):
        x_low, x_high = X_BINS[x_index]
        for bin_index, value in enumerate(item["values"]):
            stat = value * item["stat_percent"][bin_index] / 100.0
            sys_error = value * item["sys_percent"][bin_index] / 100.0
            h1_rows.append(
                {
                    "region": region,
                    "x_bin": x_index,
                    "xB_low": x_low,
                    "xB_high": x_high,
                    "pt_low": item["edges"][bin_index],
                    "pt_high": item["edges"][bin_index + 1],
                    "h1_density": value,
                    "h1_stat_error": stat,
                    "h1_sys_error": sys_error,
                    "h1_total_error": math.hypot(stat, sys_error),
                }
            )

h1_table = pd.DataFrame(h1_rows)
if len(h1_table) != 49:
    raise ValueError(f"Expected 49 H1 points, received {len(h1_table)}.")
display(h1_table.head(12))
print(f"Encoded {len(h1_table)} H1 points from Tables 8 and 9.")


,region,x_bin,xB_low,xB_high,pt_low,pt_high,h1_density,h1_stat_error,h1_sys_error,h1_total_error
0,central,0,0.00010,0.00024,0.2,0.4,4.760000,0.001428,0.152320,0.152327
1,central,0,0.00010,0.00024,0.4,0.6,2.920000,0.001168,0.084680,0.084688
2,central,0,0.00010,0.00024,0.6,1.0,1.144000,0.001144,0.042328,0.042343
3,central,0,0.00010,0.00024,1.0,2.0,0.195500,0.000762,0.005865,0.005914
4,central,0,0.00010,0.00024,2.0,4.0,0.014890,0.000281,0.000476,0.000553
5,central,0,0.00010,0.00024,4.0,10.0,0.000569,0.000087,0.000034,0.000094
6,central,1,0.00024,0.00050,0.2,0.4,3.990000,0.001995,0.099750,0.099770
7,central,1,0.00024,0.00050,0.4,0.6,2.530000,0.001518,0.065780,0.065798
8,central,1,0.00024,0.00050,0.6,1.0,0.994000,0.001392,0.028826,0.028860
9,central,1,0.00024,0.00050,1.0,2.0,0.161100,0.000838,0.005316,0.005382


Encoded 49 H1 points from Tables 8 and 9.


## 3. Parse and validate the schema-v4 event record

Each event header contains

$$
(Q^2,y,W^2,x_B,x_g,z,\mathbf{k}_{1T},\mathbf{k}_{2T},\ldots,n_{\rm had}),
$$

followed by $n_{\rm had}$ final-state hadrons `(PDG, px, py, pz, E)`.
A two-pass reader first allocates exact array sizes and then fills contiguous NumPy
arrays. This avoids millions of Python tuple objects and gives a fast transfer to CUDA.


In [3]:
EVENT_COLUMNS = [
    "pol", "back_to_back", "flavor", "Q2", "y", "W2", "xB", "xg",
    "z", "k1x", "k1y", "k2x", "k2y", "eta1", "eta2", "nHad",
]
FILE_COLUMNS = ["EV", *EVENT_COLUMNS]


def parse_key_values(line, prefix, cast=float):
    """Parse a metadata line containing whitespace-separated key=value tokens."""
    return {
        key: cast(value)
        for key, value in (
            token.split("=", maxsplit=1)
            for token in line.removeprefix(prefix).split()
        )
    }


def file_sha256(path, block_size=8 * 1024 * 1024):
    """Return a stable identity for the event file or model artifact."""
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()


def scan_event_file(path):
    """Count records and collect schema, sampling, and generation metadata."""
    metadata = {
        "schema_version": None,
        "event_semantics": None,
        "file_columns": None,
        "sampling": None,
        "support": None,
        "generation": {},
    }
    n_events = 0
    n_hadrons = 0
    with path.open("r", encoding="utf-8") as handle:
        for raw_line in handle:
            line = raw_line.strip()
            if line.startswith("# DIS dijet events"):
                match = re.search(r"schema_version=(\d+)", line)
                metadata["schema_version"] = int(match.group(1)) if match else None
            elif line.startswith("# event_semantics:"):
                metadata["event_semantics"] = parse_key_values(
                    line, "# event_semantics:", cast=str
                )
            elif line.startswith("# columns:"):
                metadata["file_columns"] = line.removeprefix("# columns:").split()
            elif line.startswith("# sampling:"):
                metadata["sampling"] = parse_key_values(line, "# sampling:")
            elif line.startswith("# inelastic_emulator_support:"):
                metadata["support"] = parse_key_values(
                    line, "# inelastic_emulator_support:"
                )
            elif line.startswith("# generation:"):
                values = parse_key_values(line, "# generation:", cast=str)
                key = (values["mode"], int(values["pol"]))
                metadata["generation"][key] = {
                    "written": int(values["written"]),
                    "tries": int(values["tries"]),
                }
            elif line.startswith("EV "):
                fields = line.split()
                if len(fields) != len(FILE_COLUMNS):
                    raise ValueError(f"Malformed event header with {len(fields)} fields.")
                n_events += 1
                n_hadrons += int(fields[-1])
    return n_events, n_hadrons, metadata


def read_event_file(path):
    """Read event headers and final-state hadrons into contiguous arrays."""
    n_events, n_hadrons, metadata = scan_event_file(path)
    event_values = np.empty((n_events, len(EVENT_COLUMNS)), dtype=np.float64)
    hadrons = np.empty((n_hadrons, 5), dtype=np.float64)
    offsets = np.empty(n_events + 1, dtype=np.int64)
    declared = np.empty(n_events, dtype=np.int64)
    event_cursor = 0
    hadron_cursor = 0

    with path.open("r", encoding="utf-8") as handle:
        for raw_line in handle:
            line = raw_line.strip()
            if not line or line.startswith("#"):
                continue
            if line.startswith("EV "):
                values = np.fromstring(line[3:], sep=" ", dtype=np.float64)
                if values.size != len(EVENT_COLUMNS):
                    raise ValueError("Unexpected event-header width.")
                offsets[event_cursor] = hadron_cursor
                event_values[event_cursor] = values
                declared[event_cursor] = int(values[-1])
                event_cursor += 1
            else:
                values = np.fromstring(line, sep=" ", dtype=np.float64)
                if values.size != 5:
                    raise ValueError("Unexpected hadron-record width.")
                hadrons[hadron_cursor] = values
                hadron_cursor += 1

    offsets[-1] = hadron_cursor
    if event_cursor != n_events or hadron_cursor != n_hadrons:
        raise ValueError("Record counts changed between parsing passes.")
    if not np.array_equal(np.diff(offsets), declared):
        raise ValueError("At least one event has an inconsistent hadron count.")

    events = pd.DataFrame(event_values, columns=EVENT_COLUMNS)
    events.insert(0, "event_id", np.arange(n_events, dtype=np.int64))
    for column in ("pol", "back_to_back", "flavor", "nHad"):
        events[column] = events[column].astype(np.int64)
    hadron_event = np.repeat(np.arange(n_events, dtype=np.int64), declared)
    return events, hadrons, hadron_event, offsets, metadata


parse_start = time.perf_counter()
events, hadrons, hadron_event, hadron_offsets, metadata = read_event_file(EVENT_FILE)
parse_seconds = time.perf_counter() - parse_start
sampling = metadata["sampling"]
generation = metadata["generation"]

if metadata["schema_version"] != 4:
    raise ValueError(f"Expected schema version 4, received {metadata['schema_version']}.")
if metadata["file_columns"] != FILE_COLUMNS:
    raise ValueError("The serialized event columns do not match this notebook.")
if metadata["event_semantics"] != {
    "quark_header": "pre_shower",
    "hadrons": "post_pythia_final_state",
}:
    raise ValueError(f"Unexpected event semantics: {metadata['event_semantics']}")
required_sampling = {
    "Q2_min": 5.0,
    "Q2_max": 10.0,
    "y_min": 0.05,
    "y_max": 0.6,
    "E_lepton": E_ELECTRON,
    "E_nucleon": E_PROTON,
    "output_frame": 1.0,
    "shower_on": 1.0,
}
for key, expected in required_sampling.items():
    if not np.isclose(sampling[key], expected):
        raise ValueError(f"Unexpected sampling metadata {key}={sampling[key]}.")
expected_strata = {
    ("inelastic", 0), ("inelastic", 1),
    ("coherent", 0), ("coherent", 1),
}
if set(generation) != expected_strata:
    raise ValueError(f"Unexpected generation strata: {sorted(generation)}")

event_digest = file_sha256(EVENT_FILE)
parse_audit = pd.DataFrame(
    {
        "quantity": [
            "events", "final-state hadrons", "parse time [s]",
            "schema version", "event-file SHA256",
        ],
        "value": [
            len(events), len(hadrons), parse_seconds,
            metadata["schema_version"], event_digest,
        ],
    }
)
display(parse_audit)
display(events.head())


,quantity,value
0,events,120000
1,final-state hadrons,2493307
2,parse time [s],3.706771
3,schema version,4
4,event-file SHA256,3af6c194d222608650ee347a2909c7ce41813aced480ff...


,event_id,pol,back_to_back,flavor,Q2,y,W2,xB,xg,z,k1x,k1y,k2x,k2y,eta1,eta2,nHad
0,0,0,0,1,9.098610,0.183151,18594.060099,0.000489,0.086803,0.367373,-16.440176,14.312380,12.426519,-6.570320,0.832086,1.814286,16
1,1,0,0,1,8.999680,0.231045,23458.618318,0.000384,0.171653,0.156412,-15.543748,-18.230774,9.006708,14.620140,-0.000088,2.018102,12
2,2,0,0,3,7.274029,0.357129,36266.515217,0.000201,0.026591,0.738744,-1.720800,12.375411,7.485943,-11.783059,2.421211,1.270854,12
3,3,0,0,1,6.019133,0.524559,53273.246029,0.000113,0.018349,0.767245,12.585297,2.143566,-13.064697,-2.452822,2.629801,1.396576,18
4,4,0,0,2,9.172318,0.265575,26965.657577,0.000340,0.111518,0.235035,-8.345651,22.645860,8.894264,-17.864696,0.469454,1.839655,19


## 4. Audit $x_B$, $x_g$, topology, and the hard tuple

The serialized gluon fraction is independently checked with

$$
x_g = \frac{Q^2+k_{1T}^2/z+k_{2T}^2/(1-z)}{W^2+Q^2-M_p^2}.
$$

The H1 bins are then defined using the separate column $x_B$. Inelastic points with a
zero transverse scale are excluded because the emulator uses polar variables
$(k_{1T},q_T,\theta)$ and those angles are undefined at zero.


In [4]:
events["channel"] = np.where(events["back_to_back"].eq(1), "coherent", "inelastic")
hard_k1 = np.hypot(events["k1x"], events["k1y"])
hard_k2 = np.hypot(events["k2x"], events["k2y"])
hard_q = np.hypot(
    events["k1x"] + events["k2x"],
    events["k1y"] + events["k2y"],
)
eps_f = np.sqrt(events["z"] * (1.0 - events["z"]) * events["Q2"])
hard_denominator = events["W2"] + events["Q2"] - M_PROTON**2
xg_recomputed = (
    events["Q2"]
    + hard_k1**2 / events["z"]
    + hard_k2**2 / (1.0 - events["z"])
) / hard_denominator
if not np.allclose(events["xg"], xg_recomputed, rtol=1.0e-12, atol=0.0):
    raise ValueError("Serialized xg is inconsistent with the hard tuple.")

coherent = events["channel"].eq("coherent").to_numpy()
inelastic = ~coherent
inelastic_supported = (
    inelastic
    & events["xg"].gt(0.0).to_numpy()
    & np.asarray(eps_f.between(0.1, 26.0, inclusive="both"))
    & np.asarray(hard_k1.between(0.1, 40.0, inclusive="both"))
    & np.asarray(hard_q.between(0.1, 40.0, inclusive="both"))
)
model_domain = coherent | inelastic_supported

h1_event_phase_space = (
    events["Q2"].between(*H1_Q2_RANGE, inclusive="neither")
    & events["y"].between(*H1_Y_RANGE, inclusive="neither")
    & events["xB"].between(X_BINS[0][0], X_BINS[-1][1], inclusive="left")
).to_numpy()

x_bin_counts = []
for index, (x_low, x_high) in enumerate(X_BINS):
    selected = (
        h1_event_phase_space
        & events["xB"].ge(x_low).to_numpy()
        & events["xB"].lt(x_high).to_numpy()
    )
    x_bin_counts.append(
        {"x_bin": index, "xB_low": x_low, "xB_high": x_high,
         "generated_events": int(selected.sum())}
    )

variable_audit = pd.DataFrame(
    {
        "quantity": [
            "minimum xB", "maximum xB", "minimum xg", "maximum xg",
            "median xg/xB", "events with xg > 0.01",
            "inelastic events", "inelastic emulator-domain events",
            "maximum xg reconstruction residual",
            "maximum coherent hard imbalance [GeV]",
        ],
        "value": [
            events["xB"].min(), events["xB"].max(),
            events["xg"].min(), events["xg"].max(),
            np.median(events["xg"] / events["xB"]),
            int((events["xg"] > XG_CROSS_SECTION_CAP).sum()),
            int(inelastic.sum()), int(inelastic_supported.sum()),
            np.max(np.abs(events["xg"] - xg_recomputed)),
            np.max(hard_q[coherent]),
        ],
    }
)
display(variable_audit)
display(pd.DataFrame(x_bin_counts))


,quantity,value
0,minimum xB,8.249629e-05
1,maximum xB,1.951924e-03
2,minimum xg,1.139675e-04
3,maximum xg,9.997131e-01
4,median xg/xB,2.264720e+02
5,events with xg > 0.01,1.094370e+05
6,inelastic events,6.000000e+04
7,inelastic emulator-domain events,5.999200e+04
8,maximum xg reconstruction residual,4.440892e-16
9,maximum coherent hard imbalance [GeV],0.000000e+00


,x_bin,xB_low,xB_high,generated_events
0,0,0.00010,0.00024,61930
1,1,0.00024,0.00050,34959
2,2,0.00050,0.00200,20499


## 5. Evaluate hard cross sections with $x_g$ and form Monte Carlo weights

The hard routines are evaluated with the event's **$x_g$**, never with $x_B$. Following
the validated prescription in `reweight.ipynb`, the model-domain input and continuation
are

$$
\widetilde{x}_g=\min(x_g,0.01),\qquad
L(x_g)=\begin{cases}
1,&x_g\le 0.01,\\
\left(\dfrac{1-x_g}{1-0.01}\right)^4,&x_g>0.01,
\end{cases}
$$

$$
d\sigma_i=d\sigma_i(\widetilde{x}_g)L(x_g).
$$

For a uniform proposal in each topology/polarization stratum,

$$
w_i=\frac{d\sigma_i\,V_c}{p_f N_{{\rm tries},c,p}},\qquad
V_{\rm coh}=\Delta Q^2\Delta y\Delta z\,(\pi k_{\max}^2),\quad
V_{\rm inel}=\Delta Q^2\Delta y\Delta z\,(\pi k_{\max}^2)^2.
$$

The cache is accepted only when the event SHA256, model identities, quadrature order,
and continuation prescription all match.


In [5]:
if str(DIS_DIR) not in sys.path:
    sys.path.insert(0, str(DIS_DIR))

from corss_section import (
    DIJET_EMULATOR_CHECKPOINT,
    DIJET_EMULATOR_SOURCE,
    el_cross_section,
    elastic_dipole_device,
    inel_cross_section,
    inelastic_emulator_backend_id,
    inelastic_emulator_device,
)

dipole_artifacts = [
    PROJECT_DIR / "dipole_function.py",
    PROJECT_DIR / "model_ciBK_best.pt",
]
for artifact in dipole_artifacts:
    if not artifact.is_file():
        raise FileNotFoundError(artifact)
elastic_backend_id = "elastic:" + ":".join(file_sha256(path) for path in dipole_artifacts)
inelastic_backend_id = inelastic_emulator_backend_id()

cache_path = OUTPUT_DIR / "h1_event_weights.npz"
cache_metadata_path = OUTPUT_DIR / "h1_event_weights_metadata.json"
expected_cache_metadata = {
    "event_sha256": event_digest,
    "event_count": len(events),
    "elastic_backend_id": elastic_backend_id,
    "inelastic_backend_id": inelastic_backend_id,
    "cross_section_n_radial": int(os.environ["CROSS_SECTION_N_RADIAL"]),
    "xg_cap": XG_CROSS_SECTION_CAP,
    "large_x_power": LARGE_X_SUPPRESSION_POWER,
}

cache_valid = False
if cache_path.is_file() and cache_metadata_path.is_file():
    cached_metadata = json.loads(cache_metadata_path.read_text(encoding="utf-8"))
    cache_valid = cached_metadata == expected_cache_metadata

event_xg = events["xg"].to_numpy(dtype=np.float64)
xg_cross_section_input = np.minimum(event_xg, XG_CROSS_SECTION_CAP)
large_x_suppression = np.ones(len(events), dtype=np.float64)
large_x_mask = event_xg > XG_CROSS_SECTION_CAP
large_x_suppression[large_x_mask] = (
    (1.0 - event_xg[large_x_mask]) / (1.0 - XG_CROSS_SECTION_CAP)
) ** LARGE_X_SUPPRESSION_POWER


def evaluate_batches(indices, function, argument_arrays, output, label):
    """Evaluate one channel in bounded batches and retain only CPU results."""
    n_idx = len(indices)
    if n_idx == 0:
        print(f"  {label}: no events")
        return
    for start in range(0, n_idx, WEIGHT_BATCH_SIZE):
        stop = min(start + WEIGHT_BATCH_SIZE, n_idx)
        batch_indices = indices[start:stop]
        arguments = [
            np.asarray(values[batch_indices], dtype=np.float64)
            for values in argument_arrays
        ]
        batch_result = function(*arguments)
        output[batch_indices] = np.asarray(batch_result, dtype=np.float64)
        del batch_result, arguments, batch_indices
        release_cuda_memory()
        if stop == n_idx or stop % (5 * WEIGHT_BATCH_SIZE) == 0:
            print(f"  {label}: {stop:,}/{n_idx:,}")


cross_section_start = time.perf_counter()
dipole_device = "not loaded (cache reused)"
emulator_device = "not loaded (cache reused)"
if cache_valid:
    with np.load(cache_path, allow_pickle=False) as cached:
        if not np.array_equal(cached["event_id"], events["event_id"].to_numpy()):
            raise ValueError("Cached event IDs do not match the current event file.")
        if not np.allclose(cached["xg_input"], xg_cross_section_input, rtol=0.0, atol=0.0):
            raise ValueError("Cached xg inputs do not match the current event file.")
        dsigma_at_capped_xg = cached["dsigma_at_capped_xg"]
        dsigma_evaluated = cached["dsigma_evaluated"]
        mc_weight = cached["mc_weight"]
    cache_status = "reused"
else:
    if DEVICE.type == "cuda":
        free_b, total_b = torch.cuda.mem_get_info(DEVICE)
        print(
            f"CUDA free/total before cross sections: "
            f"{free_b / (1024**3):.2f}/{total_b / (1024**3):.2f} GiB"
        )
        if free_b < 1.0 * (1024**3):
            warnings.warn(
                "Less than 1 GiB CUDA memory is free. Close other GPU processes "
                "(nvidia-smi) or shrink H1_WEIGHT_BATCH_SIZE / "
                "CROSS_SECTION_DIPOLE_BATCH_SIZE before continuing.",
                RuntimeWarning,
            )

    dsigma_at_capped_xg = np.zeros(len(events), dtype=np.float64)
    coherent_indices = np.flatnonzero(coherent)
    inelastic_indices = np.flatnonzero(inelastic_supported)

    print("Evaluating coherent (elastic) cross sections in batches:")
    dipole_device = elastic_dipole_device()
    evaluate_batches(
        coherent_indices,
        el_cross_section,
        [
            events["pol"].to_numpy(dtype=np.float64),
            events["y"].to_numpy(dtype=np.float64),
            events["Q2"].to_numpy(dtype=np.float64),
            events["z"].to_numpy(dtype=np.float64),
            events["k1x"].to_numpy(dtype=np.float64),
            events["k1y"].to_numpy(dtype=np.float64),
            xg_cross_section_input,
            events["flavor"].to_numpy(dtype=np.float64),
        ],
        dsigma_at_capped_xg,
        "elastic",
    )
    release_cuda_memory(move_dipole_to_cpu=True)

    print("Evaluating inelastic cross sections in batches:")
    emulator_device = inelastic_emulator_device()
    evaluate_batches(
        inelastic_indices,
        inel_cross_section,
        [
            events["pol"].to_numpy(dtype=np.float64),
            events["y"].to_numpy(dtype=np.float64),
            events["Q2"].to_numpy(dtype=np.float64),
            events["z"].to_numpy(dtype=np.float64),
            events["k1x"].to_numpy(dtype=np.float64),
            events["k1y"].to_numpy(dtype=np.float64),
            events["k2x"].to_numpy(dtype=np.float64),
            events["k2y"].to_numpy(dtype=np.float64),
            xg_cross_section_input,
            events["flavor"].to_numpy(dtype=np.float64),
        ],
        dsigma_at_capped_xg,
        "inelastic",
    )
    release_cuda_memory(move_emulator_to_cpu=True)

    dsigma_evaluated = dsigma_at_capped_xg * large_x_suppression

    base_volume = (
        (sampling["Q2_max"] - sampling["Q2_min"])
        * (sampling["y_max"] - sampling["y_min"])
        * (sampling["z_max"] - sampling["z_min"])
    )
    disk_area = np.pi * sampling["k_max"] ** 2
    phase_space_volume = {
        "coherent": base_volume * disk_area,
        "inelastic": base_volume * disk_area**2,
    }
    proposal_tries = np.empty(len(events), dtype=np.float64)
    proposal_volume = np.empty(len(events), dtype=np.float64)
    for (channel, pol), statistics in generation.items():
        selected = (
            events["channel"].eq(channel).to_numpy()
            & events["pol"].eq(pol).to_numpy()
        )
        proposal_tries[selected] = statistics["tries"]
        proposal_volume[selected] = phase_space_volume[channel]
    if sampling["flavor_pdf"] <= 0.0:
        raise ValueError("The flavor proposal density must be positive.")
    dsigma_used = np.where(model_domain, dsigma_evaluated, 0.0)
    mc_weight = (
        dsigma_used * proposal_volume
        / (sampling["flavor_pdf"] * proposal_tries)
    )
    np.savez_compressed(
        cache_path,
        event_id=events["event_id"].to_numpy(),
        xg_input=xg_cross_section_input,
        dsigma_at_capped_xg=dsigma_at_capped_xg,
        dsigma_evaluated=dsigma_evaluated,
        mc_weight=mc_weight,
    )
    cache_metadata_path.write_text(
        json.dumps(expected_cache_metadata, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )
    del dsigma_used, proposal_tries, proposal_volume, coherent_indices, inelastic_indices
    release_cuda_memory(move_dipole_to_cpu=True, move_emulator_to_cpu=True)
    cache_status = "computed and cached"

cross_section_seconds = time.perf_counter() - cross_section_start
if np.any(~np.isfinite(dsigma_at_capped_xg)) or np.any(dsigma_at_capped_xg < 0.0):
    raise ValueError("Cross sections at capped xg must be finite and nonnegative.")
if np.any(~np.isfinite(dsigma_evaluated)) or np.any(dsigma_evaluated < 0.0):
    raise ValueError("Continued cross sections must be finite and nonnegative.")
if np.any(~np.isfinite(mc_weight)) or np.any(mc_weight < 0.0):
    raise ValueError("Monte Carlo weights must be finite and nonnegative.")

events["xg_cross_section_input"] = xg_cross_section_input
events["large_x_suppression"] = large_x_suppression
events["cross_section"] = dsigma_evaluated
events["mc_weight"] = mc_weight
stratum_keys = [
    ("coherent", 0), ("coherent", 1),
    ("inelastic", 0), ("inelastic", 1),
]
stratum_lookup = {key: index for index, key in enumerate(stratum_keys)}
events["stratum"] = [
    stratum_lookup[(channel, pol)]
    for channel, pol in zip(events["channel"], events["pol"])
]
tries_by_stratum = np.asarray(
    [generation[key]["tries"] for key in stratum_keys], dtype=np.int64
)

channel_summary = (
    events.groupby(["channel", "pol"], sort=True)
    .agg(
        events=("event_id", "size"),
        positive_weight_events=("mc_weight", lambda values: int((values > 0.0).sum())),
        integrated_cross_section=("mc_weight", "sum"),
        maximum_event_weight=("mc_weight", "max"),
    )
    .reset_index()
)
display(channel_summary)
print(f"Cross-section cache: {cache_status} ({cross_section_seconds:.2f} s)")
print(f"Elastic dipole device during evaluation: {dipole_device}")
print(f"Inelastic emulator device during evaluation: {emulator_device}")
print(f"Events evaluated at capped xg: {int(large_x_mask.sum())}")
if DEVICE.type == "cuda":
    print(
        "GPU allocated after cross sections: "
        f"{torch.cuda.memory_allocated(DEVICE) / (1024**2):.1f} MiB"
    )
    free_b, total_b = torch.cuda.mem_get_info(DEVICE)
    print(
        f"CUDA free/total after cross sections: "
        f"{free_b / (1024**3):.2f}/{total_b / (1024**3):.2f} GiB"
    )


CUDA free/total before cross sections: 2.06/15.47 GiB
Evaluating coherent (elastic) cross sections in batches:
Running on the GPU
  elastic: 50,000/60,000
  elastic: 60,000/60,000
Evaluating inelastic cross sections in batches:
Running on the GPU
  inelastic: 50,000/59,992
  inelastic: 59,992/59,992


,channel,pol,events,positive_weight_events,integrated_cross_section,maximum_event_weight
0,coherent,0,30000,30000,8.497999e-08,3.410468e-08
1,coherent,1,30000,30000,1.230510e-08,2.870078e-09
2,inelastic,0,30000,29996,1.857965e-06,1.336029e-06
3,inelastic,1,30000,29996,7.766244e-08,1.183493e-08


Cross-section cache: computed and cached (32.83 s)
Elastic dipole device during evaluation: cuda:0
Inelastic emulator device during evaluation: cuda:0
Events evaluated at capped xg: 109437
GPU allocated after cross sections: 64.0 MiB
CUDA free/total after cross sections: 2.08/15.47 GiB


## 6. Reconstruct the HERA laboratory frame and apply particle-level coordinates

The event hadrons are stored in the $\gamma^*p$ centre-of-mass frame with the photon
along $+z^*$. The H1 particle phase space also requires laboratory-frame cuts, so the
Born-level virtual-photon four-vector is reconstructed from $Q^2$ and $y$.

With the H1 laboratory convention $P=(E_p,0,0,+p_p)$ and
$k=(E_e,0,0,-E_e)$,

$$
q^0+q^z=-\frac{Q^2}{2E_e},\qquad P\cdot q=y\,P\cdot k,
\qquad q^2=-Q^2.
$$

These equations determine $(q^0,q^x,q^z)$ after choosing the scattering plane. The
event-dependent Lorentz boost maps each stored HCM hadron into the collider laboratory
frame. CUDA performs this transformation for the full flattened hadron array.


In [6]:
TORCH_DTYPE = torch.float64
# Process hadrons in chunks so the full ~O(10^8) float64 table never sits on the GPU.
HADRON_CHUNK = 1_000_000


def boost_to_rest(energy, momentum, beta):
    """Boost lab-frame four-vectors to the rest frame moving with beta."""
    beta2 = torch.sum(beta * beta, dim=-1)
    gamma = torch.rsqrt(1.0 - beta2)
    beta_dot_p = torch.sum(beta * momentum, dim=-1)
    energy_rest = gamma * (energy - beta_dot_p)
    factor = ((gamma - 1.0) * beta_dot_p / beta2) - gamma * energy
    momentum_rest = momentum + factor[:, None] * beta
    return energy_rest, momentum_rest


def boost_from_rest(energy, momentum, beta):
    """Boost rest-frame four-vectors into the lab frame moving with beta."""
    beta2 = torch.sum(beta * beta, dim=-1)
    gamma = torch.rsqrt(1.0 - beta2)
    beta_dot_p = torch.sum(beta * momentum, dim=-1)
    energy_lab = gamma * (energy + beta_dot_p)
    factor = ((gamma - 1.0) * beta_dot_p / beta2) + gamma * energy
    momentum_lab = momentum + factor[:, None] * beta
    return energy_lab, momentum_lab


transform_start = time.perf_counter()
q2_t = torch.as_tensor(
    events["Q2"].to_numpy(copy=True), dtype=TORCH_DTYPE, device=DEVICE
)
y_t = torch.as_tensor(
    events["y"].to_numpy(copy=True), dtype=TORCH_DTYPE, device=DEVICE
)
p_proton = math.sqrt(E_PROTON**2 - M_PROTON**2)
q_plus = -q2_t / (2.0 * E_ELECTRON)
q0 = y_t * E_ELECTRON + p_proton * q_plus / (E_PROTON + p_proton)
qz = q_plus - q0
qx2 = q0**2 - qz**2 + q2_t
if torch.min(qx2).item() < -1.0e-10:
    raise ValueError("The reconstructed virtual photon has an invalid transverse momentum.")
qx = torch.sqrt(torch.clamp(qx2, min=0.0))

zeros = torch.zeros_like(qx)
photon_lab_p = torch.stack((qx, zeros, qz), dim=1)
proton_lab_p = torch.stack((zeros, zeros, torch.full_like(qx, p_proton)), dim=1)
total_lab_energy = q0 + E_PROTON
total_lab_p = photon_lab_p + proton_lab_p
beta_cm = total_lab_p / total_lab_energy[:, None]

total_cm_energy, total_cm_p = boost_to_rest(total_lab_energy, total_lab_p, beta_cm)
photon_cm_energy, photon_cm_p = boost_to_rest(q0, photon_lab_p, beta_cm)
ez = photon_cm_p / torch.linalg.vector_norm(photon_cm_p, dim=1)[:, None]
ey = torch.zeros_like(ez)
ey[:, 1] = 1.0
ex = torch.linalg.cross(ey, ez, dim=1)

# Drop kinematics that are not needed for the hadron loop.
del (
    q2_t,
    y_t,
    q_plus,
    q0,
    qz,
    qx2,
    qx,
    zeros,
    photon_lab_p,
    proton_lab_p,
    total_lab_energy,
    total_lab_p,
    photon_cm_energy,
    photon_cm_p,
)
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

n_hadrons = int(hadrons.shape[0])
hadron_event_t = torch.as_tensor(hadron_event, dtype=torch.int64)
pt_star = torch.empty(n_hadrons, dtype=TORCH_DTYPE)
eta_star = torch.empty(n_hadrons, dtype=TORCH_DTYPE)
pt_lab = torch.empty(n_hadrons, dtype=TORCH_DTYPE)
eta_lab = torch.empty(n_hadrons, dtype=TORCH_DTYPE)
charged_hadron = torch.empty(n_hadrons, dtype=torch.bool)

charged_ids = torch.tensor(
    [211, 321, 2212, 3112, 3222, 3312, 3334],
    dtype=torch.int64,
    device=DEVICE,
)
invariant_residual = 0.0
for start in range(0, n_hadrons, HADRON_CHUNK):
    end = min(start + HADRON_CHUNK, n_hadrons)
    hadron_chunk = torch.as_tensor(
        hadrons[start:end, 1:5], dtype=TORCH_DTYPE, device=DEVICE
    )
    hadron_id_chunk = torch.as_tensor(
        hadrons[start:end, 0], dtype=torch.int64, device=DEVICE
    )
    event_chunk = torch.as_tensor(
        hadron_event[start:end], dtype=torch.int64, device=DEVICE
    )

    hcm_p = hadron_chunk[:, :3]
    hcm_energy = hadron_chunk[:, 3]
    pt_star_chunk = torch.hypot(hcm_p[:, 0], hcm_p[:, 1])
    eta_star_chunk = torch.asinh(
        hcm_p[:, 2] / torch.clamp(pt_star_chunk, min=1.0e-15)
    )
    cm_p_lab_axes = (
        hcm_p[:, 0, None] * ex[event_chunk]
        + hcm_p[:, 1, None] * ey[event_chunk]
        + hcm_p[:, 2, None] * ez[event_chunk]
    )
    lab_energy, lab_p_chunk = boost_from_rest(
        hcm_energy, cm_p_lab_axes, beta_cm[event_chunk]
    )
    pt_lab_chunk = torch.hypot(lab_p_chunk[:, 0], lab_p_chunk[:, 1])
    eta_lab_chunk = torch.asinh(
        lab_p_chunk[:, 2] / torch.clamp(pt_lab_chunk, min=1.0e-15)
    )
    charged_chunk = torch.isin(torch.abs(hadron_id_chunk), charged_ids)

    mass2_hcm = hcm_energy**2 - torch.sum(hcm_p * hcm_p, dim=1)
    mass2_lab = lab_energy**2 - torch.sum(lab_p_chunk * lab_p_chunk, dim=1)
    invariant_residual = max(
        invariant_residual,
        torch.max(torch.abs(mass2_lab - mass2_hcm)).item(),
    )

    pt_star[start:end] = pt_star_chunk.cpu()
    eta_star[start:end] = eta_star_chunk.cpu()
    pt_lab[start:end] = pt_lab_chunk.cpu()
    eta_lab[start:end] = eta_lab_chunk.cpu()
    charged_hadron[start:end] = charged_chunk.cpu()

    del (
        hadron_chunk,
        hadron_id_chunk,
        event_chunk,
        hcm_p,
        hcm_energy,
        pt_star_chunk,
        eta_star_chunk,
        cm_p_lab_axes,
        lab_energy,
        lab_p_chunk,
        pt_lab_chunk,
        eta_lab_chunk,
        charged_chunk,
        mass2_hcm,
        mass2_lab,
    )
    # Return free blocks to the driver periodically, not every chunk.
    if DEVICE.type == "cuda" and ((start // HADRON_CHUNK) % 4 == 3 or end >= n_hadrons):
        torch.cuda.empty_cache()

cm_momentum_residual = torch.max(torch.linalg.vector_norm(total_cm_p, dim=1)).item()
w2_reconstructed = total_cm_energy**2
w2_residual = torch.max(
    torch.abs(
        w2_reconstructed
        - torch.as_tensor(
            events["W2"].to_numpy(copy=True), dtype=TORCH_DTYPE, device=DEVICE
        )
    )
).item()

# Hadron coordinates now live on CPU; free all remaining event-level GPU tensors.
del (
    beta_cm,
    total_cm_energy,
    total_cm_p,
    ex,
    ey,
    ez,
    charged_ids,
    w2_reconstructed,
)
if DEVICE.type == "cuda":
    torch.cuda.synchronize(DEVICE)
    torch.cuda.empty_cache()
transform_seconds = time.perf_counter() - transform_start

transform_audit = pd.DataFrame(
    {
        "quantity": [
            "charged final-state hadrons",
            "hadron chunk size",
            "maximum CM total-momentum residual [GeV]",
            "maximum W2 reconstruction residual [GeV2]",
            "maximum hadron mass2 invariance residual [GeV2]",
            "coordinate-transform time [s]",
            "GPU allocated after transform [MiB]",
        ],
        "value": [
            int(charged_hadron.sum().item()),
            HADRON_CHUNK,
            cm_momentum_residual,
            w2_residual,
            invariant_residual,
            transform_seconds,
            (
                torch.cuda.memory_allocated(DEVICE) / (1024**2)
                if DEVICE.type == "cuda"
                else 0.0
            ),
        ],
    }
)
display(transform_audit)


,quantity,value
0,charged final-state hadrons,1.584218e+06
1,hadron chunk size,1.000000e+06
2,maximum CM total-momentum residual [GeV],3.979060e-12
3,maximum W2 reconstruction residual [GeV2],4.656613e-10
4,maximum hadron mass2 invariance residual [GeV2],2.150955e-10
5,coordinate-transform time [s],2.638731e-01
6,GPU allocated after transform [MiB],6.400000e+01


## 7. Apply the H1 cuts and fill weighted spectra on CPU

The particle-level cuts are

$$
-2<\eta_{\rm lab}<2.5,\qquad p_{T,\rm lab}>0.15\;\mathrm{GeV},
\qquad 0<\eta^*<5,\qquad 0<p_T^*<10\;\mathrm{GeV}.
$$

For H1 bin $b$ in one $(x_B,\eta^*)$ region, the weighted estimator is

$$
\widehat{f}_b=
\frac{\sum_e w_e n_{e,b}}
     {\Delta p_{T,b}^*\sum_e w_e}.
$$

The denominator includes every selected DIS event, including events with no accepted
charged particle. The Monte Carlo error uses the ratio-estimator influence
$w_e(n_{e,b}-\widehat f_b\Delta p_{T,b}^*)$ and retains the four independent
topology/polarization proposal strata, including rejected proposals as zero
contributions through each recorded $N_{\rm tries}$.

Large selected-event matrices stay on the CPU so they cannot compete with the §6
transform for GPU memory; any leftover §6 CUDA tensors are deleted first.


In [7]:
# Histograms stay on CPU. Free any leftover §6 GPU tensors before allocating.
histogram_start = time.perf_counter()
HIST_DEVICE = torch.device("cpu")
_gpu_leftovers = [
    "q2_t",
    "y_t",
    "q_plus",
    "q0",
    "qz",
    "qx2",
    "qx",
    "zeros",
    "photon_lab_p",
    "proton_lab_p",
    "total_lab_energy",
    "total_lab_p",
    "beta_cm",
    "total_cm_energy",
    "total_cm_p",
    "photon_cm_energy",
    "photon_cm_p",
    "ex",
    "ey",
    "ez",
    "charged_ids",
    "w2_reconstructed",
]
for _name in _gpu_leftovers:
    if _name in globals():
        del globals()[_name]
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

event_weight_t = torch.as_tensor(mc_weight, dtype=TORCH_DTYPE, device=HIST_DEVICE)
event_xb_t = torch.as_tensor(
    events["xB"].to_numpy(copy=True), dtype=TORCH_DTYPE, device=HIST_DEVICE
)
event_stratum_t = torch.as_tensor(
    events["stratum"].to_numpy(copy=True), dtype=torch.int64, device=HIST_DEVICE
)
event_phase_t = torch.as_tensor(
    np.asarray(h1_event_phase_space, dtype=np.bool_),
    dtype=torch.bool,
    device=HIST_DEVICE,
)
n_events = len(events)

track_phase = (
    charged_hadron
    & (eta_lab > -2.0)
    & (eta_lab < 2.5)
    & (pt_lab > 0.15)
    & (eta_star > 0.0)
    & (eta_star < 5.0)
    & (pt_star > 0.0)
    & (pt_star < 10.0)
)
# These tensors are already on CPU; keep plain aliases for readability.
hadron_event_cpu = hadron_event_t
pt_star_cpu = pt_star
eta_star_cpu = eta_star

eta_regions = {
    "central": (0.0, 1.5),
    "current": (1.5, 5.0),
}
comparison_rows = []
selection_rows = []

for region, (eta_low, eta_high) in eta_regions.items():
    region_track = (
        track_phase & (eta_star_cpu > eta_low) & (eta_star_cpu < eta_high)
    )
    for x_index, (x_low, x_high) in enumerate(X_BINS):
        h1 = H1_MEASUREMENTS[region][x_index]
        edges_t = torch.as_tensor(
            h1["edges"], dtype=TORCH_DTYPE, device=HIST_DEVICE
        )
        widths_t = torch.diff(edges_t)
        n_bins = len(h1["values"])
        event_selected = (
            event_phase_t
            & (event_xb_t >= x_low)
            & (event_xb_t < x_high)
        )
        selected_idx = torch.nonzero(event_selected, as_tuple=False).reshape(-1)
        n_selected = int(selected_idx.numel())
        if n_selected == 0:
            raise ValueError(f"No selected events for {region}, x bin {x_index}.")

        w_selected = event_weight_t[selected_idx]
        denominator = torch.sum(w_selected)
        if denominator.item() <= 0.0:
            raise ValueError(
                f"Nonpositive event denominator for {region}, x bin {x_index}."
            )

        compact = torch.full((n_events,), -1, dtype=torch.int64, device=HIST_DEVICE)
        compact[selected_idx] = torch.arange(
            n_selected, device=HIST_DEVICE, dtype=torch.int64
        )
        counts = torch.zeros(
            (n_selected, n_bins), dtype=TORCH_DTYPE, device=HIST_DEVICE
        )

        for start in range(0, n_hadrons, HADRON_CHUNK):
            end = min(start + HADRON_CHUNK, n_hadrons)
            local_track = region_track[start:end]
            if not bool(local_track.any()):
                continue
            local_events = hadron_event_cpu[start:end][local_track]
            local_pt = pt_star_cpu[start:end][local_track]
            compact_ids = compact[local_events]
            keep = compact_ids >= 0
            if not bool(keep.any()):
                continue
            compact_ids = compact_ids[keep]
            pt_bin = torch.bucketize(local_pt[keep], edges_t, right=True) - 1
            valid = (pt_bin >= 0) & (pt_bin < n_bins)
            if not bool(valid.any()):
                continue
            flat_bin = compact_ids[valid] * n_bins + pt_bin[valid]
            counts.view(-1).index_add_(
                0,
                flat_bin,
                torch.ones(flat_bin.numel(), dtype=TORCH_DTYPE, device=HIST_DEVICE),
            )

        numerator = torch.sum(w_selected[:, None] * counts, dim=0)
        mean_count = numerator / denominator
        density = mean_count / widths_t
        influence = w_selected[:, None] * (counts - mean_count[None, :])
        stratum_selected = event_stratum_t[selected_idx]
        variance_numerator = torch.zeros(
            n_bins, dtype=TORCH_DTYPE, device=HIST_DEVICE
        )
        for stratum_index, n_tries in enumerate(tries_by_stratum):
            values = influence[stratum_selected == stratum_index]
            if values.numel() == 0:
                continue
            stratum_sum = torch.sum(values, dim=0)
            centered_sum_squares = (
                torch.sum(values * values, dim=0)
                - stratum_sum**2 / float(n_tries)
            )
            variance_numerator += (
                float(n_tries) / float(n_tries - 1)
            ) * torch.clamp(centered_sum_squares, min=0.0)
        mc_error = torch.sqrt(variance_numerator) / denominator / widths_t

        density_np = density.numpy()
        mc_error_np = mc_error.numpy()
        denominator_value = float(denominator.item())
        effective_events = (
            denominator_value**2 / float(torch.sum(w_selected**2).item())
        )
        selection_rows.append(
            {
                "region": region,
                "x_bin": x_index,
                "xB_low": x_low,
                "xB_high": x_high,
                "selected_generated_events": n_selected,
                "positive_weight_events": int(torch.sum(w_selected > 0.0).item()),
                "sum_event_weights": denominator_value,
                "weighted_effective_events": effective_events,
                "model_charged_multiplicity_in_table_bins": float(
                    torch.sum(density * widths_t).item()
                ),
            }
        )

        for bin_index, model_value in enumerate(density_np):
            h1_value = h1["values"][bin_index]
            h1_stat = h1_value * h1["stat_percent"][bin_index] / 100.0
            h1_sys = h1_value * h1["sys_percent"][bin_index] / 100.0
            h1_total = math.hypot(h1_stat, h1_sys)
            combined_error = math.hypot(h1_total, mc_error_np[bin_index])
            comparison_rows.append(
                {
                    "region": region,
                    "x_bin": x_index,
                    "xB_low": x_low,
                    "xB_high": x_high,
                    "pt_low": h1["edges"][bin_index],
                    "pt_high": h1["edges"][bin_index + 1],
                    "pt_center": 0.5 * (
                        h1["edges"][bin_index] + h1["edges"][bin_index + 1]
                    ),
                    "model_density": float(model_value),
                    "model_mc_error": float(mc_error_np[bin_index]),
                    "h1_density": h1_value,
                    "h1_stat_error": h1_stat,
                    "h1_sys_error": h1_sys,
                    "h1_total_error": h1_total,
                    "model_over_h1": float(model_value / h1_value),
                    "model_over_h1_mc_error": float(
                        mc_error_np[bin_index] / h1_value
                    ),
                    "pull_diagonal": float(
                        (model_value - h1_value) / combined_error
                    ),
                }
            )

        del (
            compact,
            counts,
            influence,
            selected_idx,
            w_selected,
            edges_t,
            widths_t,
            event_selected,
            numerator,
            mean_count,
            density,
            stratum_selected,
            variance_numerator,
            mc_error,
        )

comparison = pd.DataFrame(comparison_rows)
selection_summary = pd.DataFrame(selection_rows)
comparison_path = OUTPUT_DIR / "h1_q2_5_10_spectra.csv"
selection_path = OUTPUT_DIR / "h1_q2_5_10_selection_summary.csv"
comparison.to_csv(comparison_path, index=False)
selection_summary.to_csv(selection_path, index=False)
histogram_seconds = time.perf_counter() - histogram_start

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

display(selection_summary)
display(comparison.head(12))
minimum_effective_events = selection_summary["weighted_effective_events"].min()
if minimum_effective_events < 30.0:
    print(
        "Statistical caution: the minimum weighted effective event count is "
        f"{minimum_effective_events:.2f}; interpret weight-dominated bins with care."
    )
print(f"CPU selection/histogram time: {histogram_seconds:.2f} s")
if DEVICE.type == "cuda":
    print(
        "GPU allocated after histograms: "
        f"{torch.cuda.memory_allocated(DEVICE) / (1024**2):.1f} MiB"
    )
print(f"Saved {comparison_path}")
print(f"Saved {selection_path}")


/tmp/ipykernel_3406322/3542919281.py:41: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  event_phase_t = torch.as_tensor(


,region,x_bin,xB_low,xB_high,selected_generated_events,positive_weight_events,sum_event_weights,weighted_effective_events,model_charged_multiplicity_in_table_bins
0,central,0,0.00010,0.00024,61930,61928,1.681860e-06,1.554309,0.915497
1,central,1,0.00024,0.00050,34959,34955,1.205093e-07,7.587581,0.947312
2,central,2,0.00050,0.00200,20499,20497,2.119092e-07,4.812300,0.512624
3,current,0,0.00010,0.00024,61930,61928,1.681860e-06,1.554309,3.364216
4,current,1,0.00024,0.00050,34959,34955,1.205093e-07,7.587581,4.980898
5,current,2,0.00050,0.00200,20499,20497,2.119092e-07,4.812300,4.144319


,region,x_bin,xB_low,xB_high,pt_low,pt_high,pt_center,model_density,model_mc_error,h1_density,h1_stat_error,h1_sys_error,h1_total_error,model_over_h1,model_over_h1_mc_error,pull_diagonal
0,central,0,0.00010,0.00024,0.2,0.4,0.3,4.524208,0.437428,4.760000,0.001428,0.152320,0.152327,0.950464,0.091897,-0.509060
1,central,0,0.00010,0.00024,0.4,0.6,0.5,0.010801,0.009484,2.920000,0.001168,0.084680,0.084688,0.003699,0.003248,-34.138534
2,central,0,0.00010,0.00024,0.6,1.0,0.8,0.014454,0.015147,1.144000,0.001144,0.042328,0.042343,0.012634,0.013241,-25.117104
3,central,0,0.00010,0.00024,1.0,2.0,1.5,0.001413,0.001166,0.195500,0.000762,0.005865,0.005914,0.007225,0.005966,-32.196353
4,central,0,0.00010,0.00024,2.0,4.0,3.0,0.000509,0.000436,0.014890,0.000281,0.000476,0.000553,0.034216,0.029252,-20.419992
5,central,0,0.00010,0.00024,4.0,10.0,7.0,0.000047,0.000039,0.000569,0.000087,0.000034,0.000094,0.082792,0.068359,-5.138128
6,central,1,0.00024,0.00050,0.2,0.4,0.3,2.977975,2.099427,3.990000,0.001995,0.099750,0.099770,0.746360,0.526172,-0.481505
7,central,1,0.00024,0.00050,0.4,0.6,0.5,1.454794,0.862023,2.530000,0.001518,0.065780,0.065798,0.575018,0.340721,-1.243687
8,central,1,0.00024,0.00050,0.6,1.0,0.8,0.073751,0.033928,0.994000,0.001392,0.028826,0.028860,0.074196,0.034133,-20.660258
9,central,1,0.00024,0.00050,1.0,2.0,1.5,0.019614,0.008066,0.161100,0.000838,0.005316,0.005382,0.121751,0.050066,-14.591633


Statistical caution: the minimum weighted effective event count is 1.55; interpret weight-dominated bins with care.
CPU selection/histogram time: 4.68 s
GPU allocated after histograms: 64.0 MiB
Saved /home/congyi/work/dipole_function_saved/DIS/results/H1/h1_q2_5_10_spectra.csv
Saved /home/congyi/work/dipole_function_saved/DIS/results/H1/h1_q2_5_10_selection_summary.csv


## 8. Plot the six spectra and the model/H1 ratios

Experimental points use the quadrature sum of the published statistical and systematic
uncertainties. The model band is the stratified Monte Carlo error only. The ratio plot
shows the H1 relative-uncertainty band around one; no normalization parameter is fitted.


In [ ]:
region_labels = {
    "central": r"$0<\eta^*<1.5$",
    "current": r"$1.5<\eta^*<5$",
}
region_order = ["central", "current"]

figure, axes = plt.subplots(
    2, 3, figsize=(15.2, 8.8), sharey=True, constrained_layout=False
)
for row, region in enumerate(region_order):
    for x_index, (x_low, x_high) in enumerate(X_BINS):
        axis = axes[row, x_index]
        subset = comparison[
            (comparison["region"] == region)
            & (comparison["x_bin"] == x_index)
        ].sort_values("pt_low")
        edges = np.r_[subset["pt_low"].to_numpy(), subset["pt_high"].iloc[-1]]
        model = subset["model_density"].to_numpy()
        model_error = subset["model_mc_error"].to_numpy()
        centers = subset["pt_center"].to_numpy()
        x_error = 0.5 * (
            subset["pt_high"].to_numpy() - subset["pt_low"].to_numpy()
        )

        axis.stairs(
            model, edges, color="#c23b23", linewidth=2.0,
            label="CGC + PB-TMD + PYTHIA",
        )
        lower = np.maximum(model - model_error, np.finfo(float).tiny)
        upper = model + model_error
        axis.fill_between(
            edges,
            np.r_[lower, lower[-1]],
            np.r_[upper, upper[-1]],
            step="post",
            color="#c23b23",
            alpha=0.20,
            linewidth=0.0,
            label="model MC uncertainty",
        )
        axis.errorbar(
            centers,
            subset["h1_density"],
            xerr=x_error,
            yerr=subset["h1_total_error"],
            fmt="o",
            color="black",
            markersize=4.2,
            capsize=2.0,
            linewidth=1.0,
            label=r"H1 data (stat. $\oplus$ sys.)",
            zorder=5,
        )
        axis.set_yscale("log")
        axis.set_xlim(0.0, 10.0)
        axis.set_ylim(1.0e-5, 20.0)
        axis.set_title(
            rf"${x_low:g}<x_B<{x_high:g}$" + "\n" + region_labels[region]
        )
        if row == 1:
            axis.set_xlabel(r"$p_T^*$ [GeV]")
        if x_index == 0:
            axis.set_ylabel(r"$(1/N)\,dn/dp_T^*$ [GeV$^{-1}$]")

handles, labels = axes[0, 0].get_legend_handles_labels()
figure.subplots_adjust(
    left=0.07, right=0.99, bottom=0.08, top=0.82, wspace=0.06, hspace=0.30
)
figure.suptitle(
    r"H1 charged-particle spectra, $5<Q^2<10\;\mathrm{GeV}^2$", y=0.985
)
figure.legend(
    handles, labels, loc="upper center", bbox_to_anchor=(0.5, 0.945),
    ncol=3, frameon=False
)
comparison_png = OUTPUT_DIR / "h1_q2_5_10_pt_comparison.png"
comparison_pdf = OUTPUT_DIR / "h1_q2_5_10_pt_comparison.pdf"
figure.savefig(comparison_png, bbox_inches="tight")
figure.savefig(comparison_pdf, bbox_inches="tight")
plt.show()

metric_rows = []
for (region, x_index), subset in comparison.groupby(["region", "x_bin"], sort=False):
    metric_rows.append(
        {
            "region": region,
            "x_bin": x_index,
            "xB_low": subset["xB_low"].iloc[0],
            "xB_high": subset["xB_high"].iloc[0],
            "points": len(subset),
            "diagonal_chi2": float(np.sum(subset["pull_diagonal"] ** 2)),
            "mean_model_over_h1": float(np.mean(subset["model_over_h1"])),
        }
    )
metrics = pd.DataFrame(metric_rows)
metrics_path = OUTPUT_DIR / "h1_q2_5_10_metrics.csv"
metrics.to_csv(metrics_path, index=False)
display(metrics)
print(f"Saved {comparison_png}")
print(f"Saved {comparison_pdf}")
print(f"Saved {metrics_path}")


## 9. Final numerical checks

The final cell verifies that all 49 published points have one model prediction, that no
model value or uncertainty is non-finite, that the event denominators are positive, and
that the comparison used $x_B$ only for selections while retaining $x_g$ in the hard
cross-section cache.


In [ ]:
if len(comparison) != len(h1_table):
    raise ValueError("The model/H1 comparison does not contain all published points.")
numeric_columns = [
    "model_density", "model_mc_error", "h1_density", "h1_total_error",
    "model_over_h1", "pull_diagonal",
]
if not np.isfinite(comparison[numeric_columns].to_numpy()).all():
    raise ValueError("The final comparison contains a non-finite value.")
if (comparison["model_density"] < 0.0).any() or (comparison["model_mc_error"] < 0.0).any():
    raise ValueError("The model spectrum or its uncertainty is negative.")
if (selection_summary["sum_event_weights"] <= 0.0).any():
    raise ValueError("At least one H1 event denominator is nonpositive.")
if not np.array_equal(
    events["xg_cross_section_input"].to_numpy(),
    np.minimum(events["xg"].to_numpy(), XG_CROSS_SECTION_CAP),
):
    raise ValueError("The hard cross-section input is not derived from xg.")

output_inventory = pd.DataFrame(
    {
        "artifact": [
            comparison_path, selection_path, metrics_path,
            comparison_png, comparison_pdf,
            cache_path, cache_metadata_path,
        ],
        "exists": [path.is_file() for path in [
            comparison_path, selection_path, metrics_path,
            comparison_png, comparison_pdf,
            cache_path, cache_metadata_path,
        ]],
    }
)
display(output_inventory)
print(f"Validated {len(comparison)} model/data points in six H1 spectra.")
print("Event selection variable: xB")
print("Hard cross-section variable: xg (with the documented xg=0.01 cap)")


## Interpretation and scope

- The calculation compares absolute weighted per-event densities; it is not normalized
  to the H1 data or to unit area.
- The published $x$ cuts are applied to serialized Bjorken $x_B$. The hard functions see
  serialized $x_g$ through the explicitly documented model-domain continuation.
- The event record is already in the photon-proton HCM with the photon along $+z^*$.
  Laboratory particle cuts are reconstructed from the exact Born-level HERA beam
  kinematics using $(Q^2,y)$. This record does not contain a detector simulation,
  scattered-lepton reconstruction, vertex position, or QED-radiation observables, so
  detector-level cuts on those quantities cannot be emulated. The H1 tables are unfolded
  to the stable charged-particle phase space used here.
- The PDF provides relative systematic uncertainties but not their covariance matrix;
  the reported diagonal $\chi^2$ values should therefore not be interpreted as a formal
  goodness-of-fit probability.
